In [31]:
from functools import reduce
from pyspark.sql.functions import (col, trim, lower, regexp_replace, sum, udf, to_timestamp,split, datediff, substring, length,
    current_timestamp, when, datediff, try_to_timestamp, to_date)
from pythainlp import word_tokenize
from pyspark.sql.types import ArrayType, StringType
from pythainlp.corpus import thai_stopwords
from xgboost.spark import SparkXGBRegressor 


In [32]:
spark_url = 'local'
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType

spark = SparkSession.builder \
    .appName("XGBoost_Clean_Run") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .master("local[*]") \
    .getOrCreate()

In [33]:
sc = spark.sparkContext

from pathlib import Path
from dotenv import load_dotenv
import os
load_dotenv()

PROJECT_ROOT = Path(os.getenv("PROJECT_ROOT"))
MONGO_URI = os.getenv("MONGO_URI")
PROCRESSED_DIR = PROJECT_ROOT / "data" / "processed" / "traffy-fondue"

Traffy_File_Path = PROCRESSED_DIR/'traffy_fondue_bangkok_processed.csv'

df_processed = spark.read.csv(str(Traffy_File_Path), header=True, inferSchema=True)

print("Schema หลังโหลดข้อมูลใหม่:")
df_processed.printSchema()

Schema หลังโหลดข้อมูลใหม่:
root
 |-- ticket_id: string (nullable = true)
 |-- address: string (nullable = true)
 |-- district: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- DaysActive_Pending: integer (nullable = true)
 |-- comment_clean: string (nullable = true)
 |-- timestamp_dt: timestamp (nullable = true)
 |-- last_activity_dt: timestamp (nullable = true)
 |-- type_ถนน: integer (nullable = true)
 |-- type_ทางเท้า: integer (nullable = true)
 |-- type_ความปลอดภัย: integer (nullable = true)
 |-- type_แสงสว่าง: integer (nullable = true)
 |-- type_ความสะอาด: integer (nullable = true)
 |-- type_กีดขวาง: integer (nullable = true)
 |-- type_ท่อระบายน้ำ: integer (nullable = true)
 |-- type_น้ำท่วม: integer (nullable = true)
 |-- type_ต้นไม้: integer (nullable = true)
 |-- type_PM25: integer (nullable = true)
 |-- type_จราจร: integer (nullable = true)
 |-- type_สะพาน: integer (nullable = true)
 |-- year_reported: integer (nullable = true)

In [34]:

df_current = df_processed 

# 1. กำหนดคอลัมน์ที่ต้องแปลง
float_cols = ["lat", "lon", "DaysActive_Pending"]
int_cols = [
    "type_ถนน", "type_ทางเท้า", "type_ความปลอดภัย", "type_แสงสว่าง", 
    "type_ความสะอาด", "type_กีดขวาง", "type_ท่อระบายน้ำ", "type_น้ำท่วม", 
    "type_ต้นไม้", "type_PM25", "type_จราจร", "type_สะพาน",
    "year_reported", "year_last_activity"
]

# 2. แปลงคอลัมน์ Float และ Integer
for c in float_cols:
    df_current = df_current.withColumn(c, col(c).cast("float"))

for c in int_cols:
    df_current = df_current.withColumn(c, col(c).cast("int"))



df_ml_ready = df_current.drop("timestamp_dt", "last_activity_dt","address","comment_clean","year_reported","year_last_activity","lat","lon")

print("Columns timestamp_dt and last_activity_dt have been dropped.")

# ตรวจสอบ Schema ใหม่
print("The updated schema is:")
df_ml_ready.printSchema()
df_ml_ready.count()

Columns timestamp_dt and last_activity_dt have been dropped.
The updated schema is:
root
 |-- ticket_id: string (nullable = true)
 |-- district: string (nullable = true)
 |-- DaysActive_Pending: float (nullable = true)
 |-- type_ถนน: integer (nullable = true)
 |-- type_ทางเท้า: integer (nullable = true)
 |-- type_ความปลอดภัย: integer (nullable = true)
 |-- type_แสงสว่าง: integer (nullable = true)
 |-- type_ความสะอาด: integer (nullable = true)
 |-- type_กีดขวาง: integer (nullable = true)
 |-- type_ท่อระบายน้ำ: integer (nullable = true)
 |-- type_น้ำท่วม: integer (nullable = true)
 |-- type_ต้นไม้: integer (nullable = true)
 |-- type_PM25: integer (nullable = true)
 |-- type_จราจร: integer (nullable = true)
 |-- type_สะพาน: integer (nullable = true)



1898

In [35]:
from pyspark.sql.functions import col, trim, regexp_replace, count

# รายชื่อ 50 เขตอย่างเป็นทางการของกรุงเทพมหานคร
BANGKOK_50_DISTRICTS = [
    "คลองสาน", "คลองสามวา", "คลองเตย", "คันนายาว", "จตุจักร", "จอมทอง", 
    "ดอนเมือง", "ดินแดง", "ดุสิต", "ตลิ่งชัน", "ทวีวัฒนา", "ทุ่งครุ", 
    "ธนบุรี", "บางกอกน้อย", "บางกอกใหญ่", "บางกะปิ", "บางขุนเทียน", 
    "บางคอแหลม", "บางซื่อ", "บางนา", "บางบอน", "บางพลัด", "บางรัก", 
    "บางเขน", "บางแค", "บึงกุ่ม", "ปทุมวัน", "ประเวศ", "ป้อมปราบศัตรูพ่าย", 
    "พญาไท", "พระนคร", "พระโขนง", "ภาษีเจริญ", "มีนบุรี", "ยานนาวา", 
    "ราชเทวี", "ราษฎร์บูรณะ", "ลาดกระบัง", "ลาดพร้าว", "วังทองหลาง", 
    "วัฒนา", "สวนหลวง", "สะพานสูง", "สัมพันธวงศ์", "สาทร", "สายไหม", 
    "หนองจอก", "หนองแขม", "หลักสี่", "ห้วยขวาง"
]
# ใช้ DataFrame ข้อมูล Ticket ที่คุณใช้ในการคำนวณ Livability Score (df_ml_ready)

# 1. ทำความสะอาดอย่างเข้มงวดที่สุด
df_cleaned = df_ml_ready.withColumn("district_cleaned", col("district"))

# ลบอักขระที่มองไม่เห็นทั้งหมด (Non-printable/control characters)
# เช่น \u200b, \u0000, \u0001, ฯลฯ
df_cleaned = df_cleaned.withColumn(
    "district_cleaned",
    regexp_replace(col("district_cleaned"), "[\\p{C}]", "") 
)


# แก้ไข Typo และลบช่องว่างหลายช่องให้เหลือช่องว่างเดียว
df_cleaned = df_cleaned.withColumn(
    "district_cleaned",
    regexp_replace(col("district_cleaned"), "ป้อมปราบศัตรูพ่า", "ป้อมปราบศัตรูพ่าย")
)
df_cleaned = df_cleaned.withColumn(
    "district_cleaned", 
    regexp_replace(col("district_cleaned"), "ป้อมปราบศัตรูพ่ายย", "ป้อมปราบศัตรูพ่าย") # 💡 แก้ไข Typo ใหม่
)
df_cleaned = df_cleaned.withColumn(
    "district_cleaned",
    regexp_replace(col("district_cleaned"), "\\s+", " ")
)

# Trim สุดท้ายเพื่อลบช่องว่างหัวท้ายที่อาจหลงเหลือ
df_cleaned = df_cleaned.withColumn(
    "district_cleaned",
    trim(col("district_cleaned"))
)

# 2. ใช้ district_cleaned แทน district เดิม
df_ml_ready_filtered = df_cleaned.drop("district") \
                                 .withColumnRenamed("district_cleaned", "district")

# 3. กรองเฉพาะ 50 เขตของกรุงเทพฯ เท่านั้น
df_ml_ready_filtered = df_ml_ready_filtered.filter(
    col("district").isin(BANGKOK_50_DISTRICTS)
)

# 4. ตรวจสอบจำนวนเขตสุดท้าย
df_ml_ready_filtered.cache()
count_after_filter = df_ml_ready_filtered.count()
final_district_count = df_ml_ready_filtered.select("district").distinct().count()

df_ml_ready_filtered.select("district").distinct().show(60)
print(f"จำนวนเขตสุดท้ายที่ถูกต้อง: {final_district_count} เขต")

+-----------------+
|         district|
+-----------------+
|         ตลิ่งชัน|
|         ดอนเมือง|
|           บางรัก|
|           ประเวศ|
|           บางบอน|
|            ดุสิต|
|            พญาไท|
|          หนองจอก|
|          ราชเทวี|
|      บางขุนเทียน|
|            บางแค|
|      ราษฎร์บูรณะ|
|          คลองเตย|
|           จอมทอง|
|        คลองสามวา|
|       บางกอกใหญ่|
|          ทุ่งครุ|
|         ลาดพร้าว|
|           บางเขน|
|         ทวีวัฒนา|
|         สะพานสูง|
|           ธนบุรี|
|          บึงกุ่ม|
|           ดินแดง|
|          บางกะปิ|
|          ปทุมวัน|
|         ห้วยขวาง|
|        ลาดกระบัง|
|          บางซื่อ|
|          สวนหลวง|
|          ยานนาวา|
|           สายไหม|
|           พระนคร|
|          จตุจักร|
|            วัฒนา|
|       วังทองหลาง|
|          พระโขนง|
|            บางนา|
|        บางคอแหลม|
|         คันนายาว|
|       บางกอกน้อย|
|          มีนบุรี|
|        ภาษีเจริญ|
|          หลักสี่|
|          หนองแขม|
|      สัมพันธวงศ์|
|          คลองสาน|


In [36]:
from pyspark.sql.functions import  lit

# สมมติว่านี่คือการเริ่มต้น session ใหม่ ถ้าไม่ ให้ข้ามไปขั้นตอนถัดไป
# spark = SparkSession.builder.appName("GeoSpatialJoin").getOrCreate()
PROCRESSED_DIR = PROJECT_ROOT / "data" / "processed" / "ddproperty"

CONDO_FILE_PATH = PROCRESSED_DIR/"ddproperty_processed.csv" 
# โปรดเปลี่ยนเป็น path จริงหากคุณไม่ได้อัปโหลดไฟล์ผ่านเครื่องมือ

df_condo_raw = spark.read.csv(
    str(CONDO_FILE_PATH),
    header=True,
    encoding="UTF-8",
    inferSchema=True  # ให้ Spark ลองเดาประเภทข้อมูล
)


df_condo_raw.printSchema()


root
 |-- url: string (nullable = true)
 |-- title: string (nullable = true)
 |-- publish_date: date (nullable = true)
 |-- price: double (nullable = true)
 |-- price_per_sqm: double (nullable = true)
 |-- usable_area: double (nullable = true)
 |-- bedroom: double (nullable = true)
 |-- restroom: double (nullable = true)
 |-- coords: string (nullable = true)
 |-- full_address: string (nullable = true)
 |-- sub_district: string (nullable = true)
 |-- district: string (nullable = true)
 |-- province: string (nullable = true)
 |-- postcode: integer (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)



In [37]:
df_condo_clean = df_condo_raw.drop("url","price_per_sqm", "coords", "full_address","sub_district","province","postcode","latitude","longitude")
df_condo_clean.printSchema()
df_condo_clean.show(5)

root
 |-- title: string (nullable = true)
 |-- publish_date: date (nullable = true)
 |-- price: double (nullable = true)
 |-- usable_area: double (nullable = true)
 |-- bedroom: double (nullable = true)
 |-- restroom: double (nullable = true)
 |-- district: string (nullable = true)

+--------------------+------------+---------+-----------+-------+--------+--------+
|               title|publish_date|    price|usable_area|bedroom|restroom|district|
+--------------------+------------+---------+-----------+-------+--------+--------+
|The Old Siam Resi...|  2025-11-26|6480000.0|      100.0|    2.0|     2.0|  พระนคร|
|Juldis River Mans...|  2025-11-21|3500000.0|       37.0|    1.0|     1.0|  พระนคร|
|The Old Siam Resi...|  2025-11-20|7050000.0|      118.0|    2.0|     2.0|  พระนคร|
|Juldis River Mans...|  2025-11-20|2790000.0|       41.0|    1.0|     1.0|  พระนคร|
|The Old Siam Resi...|  2025-11-15|6320000.0|      100.0|    2.0|     2.0|  พระนคร|
+--------------------+------------+---------

In [38]:
from pyspark.sql.functions import col, regexp_replace, trim

# 1. ทำความสะอาดช่องว่างและลบ "เขต" ออก
df_condo_clean = df_condo_clean.withColumn(
    "district_cleaned", 
    trim(col("district"))
)
df_condo_clean = df_condo_clean.withColumn(
    "district_cleaned", 
    trim(col("district_cleaned")) # ลบช่องว่างที่อาจเหลือ
)


In [39]:

# 2. Drop คอลัมน์เก่าและ Rename คอลัมน์ใหม่
df_condo_clean = df_condo_clean.drop("district") \
                               .withColumnRenamed("district_cleaned", "district")



# # 4. ตรวจสอบผลลัพธ์
df_condo_clean.select("district").distinct().count()
# df_condo_clean.show(10, truncate=False)

49

In [40]:
# 1. (สำคัญ) Cache ข้อมูลก่อนเพื่อตัด Lineage ที่ซับซ้อน
# หาก df_condo_clean ผ่านการ clean มาเยอะ การ cache จะช่วยให้ไม่ timeout
df_condo_clean.cache()
print(f"จำนวนแถวข้อมูลคอนโด: {df_condo_clean.count()}") # บังคับให้ประมวลผลและ Cache

# 2. ดึงชื่อเขตที่ไม่ซ้ำกันมาเป็น List (โดยไม่ใช้ RDD)
# .collect() จะได้ List ของ Row objects [Row(district='A'), Row(district='B'), ...]
rows = df_condo_clean.select("district").distinct().collect()

# 3. แปลง Row เป็น String ด้วย Python List Comprehension
condo_districts_set = set([row['district'] for row in rows])

# 4. เปรียบเทียบกับ List มาตรฐาน 50 เขต
all_districts_set = set(BANGKOK_50_DISTRICTS)

# หาเขตที่หายไป (Set Difference)
missing_districts = all_districts_set - condo_districts_set

print(f"เขตที่ไม่มีข้อมูลคอนโด ({len(missing_districts)} เขต) คือ: {missing_districts}")



จำนวนแถวข้อมูลคอนโด: 7346
เขตที่ไม่มีข้อมูลคอนโด (1 เขต) คือ: {'ทวีวัฒนา'}


In [41]:
from pyspark.sql.functions import col, count, avg, sum

# 1. จัดกลุ่มด้วยเขต และคำนวณตัวชี้วัดปัญหา
df_district_metrics = df_ml_ready_filtered.groupBy("district").agg(
    # A. Total Problem Count (ปริมาณปัญหาทั้งหมด)
    count(col("ticket_id")).alias("Total_Problem_Count"),
    
    # B. Average Severity (ความรุนแรงเฉลี่ย: ระยะเวลารอแก้ไข)
    avg(col("DaysActive_Pending")).alias("Avg_Pending_Days"),
    
    # C. Weighted Problem Type Index (ดัชนีปัญหาเฉพาะทาง - ให้ความปลอดภัยสำคัญสุด)
    # *หมายเหตุ: type_... ต้องถูก cast เป็น Int ก่อนแล้ว
    (
        (sum(col("type_ความปลอดภัย")) * 3) + 
        (sum(col("type_ทางเท้า")) * 2) +           
        (sum(col("type_น้ำท่วม")) * 3) +
        (sum(col("type_แสงสว่าง")) * 1)  +
        (sum(col("type_กีดขวาง")) * 2) +  
        (sum(col("type_ท่อระบายน้ำ")) * 2) +   
        (sum(col("type_ความสะอาด")) * 2) +          
        (sum(col("type_ถนน")) * 2) +
        (sum(col("type_ต้นไม้")) * 1)+
        (sum(col("type_PM25")) * 2) +          
        (sum(col("type_จราจร")) * 2) +
        (sum(col("type_สะพาน")) * 1)      
    ).alias("Weighted_Problem_Index")
)

print("--- ตัวชี้วัดปัญหาต่อเขต (ก่อน Normalization) ---")
df_district_metrics.orderBy(col("Avg_Pending_Days").desc()).show(5, truncate=False)

--- ตัวชี้วัดปัญหาต่อเขต (ก่อน Normalization) ---
+-----------+-------------------+------------------+----------------------+
|district   |Total_Problem_Count|Avg_Pending_Days  |Weighted_Problem_Index|
+-----------+-------------------+------------------+----------------------+
|คันนายาว   |25                 |4.36              |45                    |
|ราษฎร์บูรณะ|15                 |4.333333333333333 |30                    |
|ปทุมวัน    |41                 |4.024390243902439 |76                    |
|บางพลัด    |86                 |3.9651162790697674|172                   |
|บางแค      |30                 |3.8               |62                    |
+-----------+-------------------+------------------+----------------------+
only showing top 5 rows


In [42]:
from pyspark.ml.feature import VectorAssembler, MinMaxScaler
from pyspark.sql.functions import lit, round

# 1. รวมคอลัมน์ปัญหาทั้งหมดเข้าด้วยกัน
feature_cols = ["Total_Problem_Count", "Avg_Pending_Days", "Weighted_Problem_Index"]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features_vector"
)
df_vector = assembler.transform(df_district_metrics)

# 2. ทำ Min-Max Scaling (ปรับค่าให้อยู่ในช่วง 0 ถึง 1)
scaler = MinMaxScaler(
    inputCol="features_vector", 
    outputCol="normalized_features"
)
scaler_model = scaler.fit(df_vector)
df_scaled = scaler_model.transform(df_vector)

# 3. ดึงค่า Normalized กลับมาเป็นคอลัมน์ (ใช้ค่าแรกใน SparseVector)
# เนื่องจากเราไม่สามารถดึงค่าจาก Vector ใน PySpark ได้ง่าย ๆ เราจะใช้เทคนิคการ Scale ซ้ำ
# หรือในทางปฏิบัติที่ง่ายกว่าคือการคำนวณสูตร Min-Max ด้วยมือ (สำหรับตอนนี้)

In [43]:
from pyspark.sql.functions import col, min, max, lit, round, min, max

# 1. คำนวณค่า Min/Max สำหรับแต่ละตัวชี้วัด (แก้ไขโดยใช้ col() หุ้มชื่อคอลัมน์)
min_max_values = df_district_metrics.select(
    min(col("Total_Problem_Count")).alias("min_count"),
    max(col("Total_Problem_Count")).alias("max_count"),
    min(col("Avg_Pending_Days")).alias("min_days"),
    max(col("Avg_Pending_Days")).alias("max_days"),
    min(col("Weighted_Problem_Index")).alias("min_weight"),
    max(col("Weighted_Problem_Index")).alias("max_weight")
).collect()[0]

# ดึงค่า Min/Max
min_count, max_count = min_max_values["min_count"], min_max_values["max_count"]
min_days, max_days = min_max_values["min_days"], min_max_values["max_days"]
min_weight, max_weight = min_max_values["min_weight"], min_max_values["max_weight"]

# 2. สร้าง "ดัชนีความไม่น่าอยู่" (Unlivability Index)
df_scored = df_district_metrics.withColumn(
    # Normalize Count
    "Norm_Count", 
    (col("Total_Problem_Count") - lit(min_count)) / lit(max_count - min_count)
).withColumn(
    # Normalize Pending Days
    "Norm_Days", 
    (col("Avg_Pending_Days") - lit(min_days)) / lit(max_days - min_days)
).withColumn(
    # Normalize Weighted Index
    "Norm_Weight", 
    (col("Weighted_Problem_Index") - lit(min_weight)) / lit(max_weight - min_weight)
)

from pyspark.sql.functions import col, lit, round

# 3. รวมดัชนีเข้าด้วยกัน
df_final_index = df_scored.withColumn(
    "Unlivability_Index",
    round(
        (col("Norm_Count") * 0.3) +        # น้ำหนัก 30%
        (col("Norm_Days") * 0.4) +         # น้ำหนัก 40%
        (col("Norm_Weight") * 0.3),        # น้ำหนัก 30%
        4
    )
)

# 4. แปลงเป็น "Livability Score" ที่มีคะแนนเต็ม 10
df_final_score = df_final_index.withColumn(
    # คำนวณ Livability Score (1.0 - Unlivability Index)
    "Livability_Score_0_1",
    lit(1.0) - col("Unlivability_Index")
).withColumn(
    # ปรับคะแนนจาก 0-1 ให้เป็น 0-10 และปัดทศนิยม 2 ตำแหน่ง
    "Livability_Score_10",
    round(col("Livability_Score_0_1") * 10, 2)
).drop("Livability_Score_0_1") # ลบคอลัมน์ 0-1 ทิ้ง

# 5. จัดอันดับเขตที่ "น่าอยู่ที่สุด" (Livability_Score_10 สูงสุด)
print("\n--- อันดับเขตที่น่าอยู่ที่สุด (Livability Score เต็ม 10) ---")
df_final_score.select(
    "district", 
    "Livability_Score_10", 
    "Total_Problem_Count",
    "Avg_Pending_Days"
).orderBy(col("Livability_Score_10").desc()).show(5, truncate=False)



--- อันดับเขตที่น่าอยู่ที่สุด (Livability Score เต็ม 10) ---
+----------+-------------------+-------------------+------------------+
|district  |Livability_Score_10|Total_Problem_Count|Avg_Pending_Days  |
+----------+-------------------+-------------------+------------------+
|ทวีวัฒนา  |9.73               |17                 |2.588235294117647 |
|ทุ่งครุ   |9.06               |22                 |2.6818181818181817|
|หนองจอก   |8.49               |25                 |2.84              |
|บางกอกใหญ่|8.23               |21                 |3.0476190476190474|
|ดอนเมือง  |8.13               |25                 |3.0               |
+----------+-------------------+-------------------+------------------+
only showing top 5 rows


In [ ]:
import pandas as pd


# 1) ดึงข้อมูลทั้งหมดจาก Spark → pandas (ไม่ตัดคอลัมน์)
pdf_score = df_final_score.toPandas()

# 2) ตารางพิกัดศูนย์กลางเขต
district_centers = [
    ("พระนคร", 13.7539, 100.5017),
    ("ดุสิต", 13.7768, 100.5170),
    ("หนองจอก", 13.8600, 100.8850),
    ("บางรัก", 13.7281, 100.5229),
    ("บางเขน", 13.8700, 100.6040),
    ("บางกะปิ", 13.7700, 100.6450),
    ("ปทุมวัน", 13.7390, 100.5320),
    ("ป้อมปราบศัตรูพ่าย", 13.7466, 100.5080),
    ("พระโขนง", 13.7150, 100.5890),
    ("มีนบุรี", 13.8063, 100.7230),
    ("ลาดกระบัง", 13.7270, 100.7780),
    ("ยานนาวา", 13.6960, 100.5420),
    ("สัมพันธวงศ์", 13.7370, 100.5140),
    ("พญาไท", 13.7800, 100.5310),
    ("ธนบุรี", 13.7200, 100.4900),
    ("บางกอกใหญ่", 13.7340, 100.4850),
    ("ห้วยขวาง", 13.7760, 100.5730),
    ("คลองสาน", 13.7260, 100.5070),
    ("ตลิ่งชัน", 13.7800, 100.4410),
    ("บางกอกน้อย", 13.7590, 100.4770),
    ("บางขุนเทียน", 13.6330, 100.4340),
    ("ภาษีเจริญ", 13.7280, 100.4570),
    ("หนองแขม", 13.7050, 100.3500),
    ("ราษฎร์บูรณะ", 13.6780, 100.4950),
    ("บางพลัด", 13.7870, 100.5050),
    ("ดินแดง", 13.7660, 100.5610),
    ("บึงกุ่ม", 13.8070, 100.6430),
    ("สาทร", 13.7160, 100.5260),
    ("บางซื่อ", 13.8180, 100.5400),
    ("จตุจักร", 13.8190, 100.5560),
    ("บางคอแหลม", 13.6930, 100.5090),
    ("ประเวศ", 13.7200, 100.7000),
    ("คลองเตย", 13.7160, 100.5680),
    ("สวนหลวง", 13.7210, 100.6300),
    ("จอมทอง", 13.6850, 100.4750),
    ("ดอนเมือง", 13.9240, 100.5890),
    ("ราชเทวี", 13.7560, 100.5400),
    ("ลาดพร้าว", 13.8080, 100.6100),
    ("วัฒนา", 13.7360, 100.5850),
    ("บางแค", 13.6940, 100.4110),
    ("หลักสี่", 13.8830, 100.5860),
    ("สายไหม", 13.9320, 100.6490),
    ("คันนายาว", 13.8300, 100.7030),
    ("สะพานสูง", 13.7700, 100.7040),
    ("วังทองหลาง", 13.7900, 100.6030),
    ("คลองสามวา", 13.8670, 100.7170),
    ("บางนา", 13.6680, 100.6140),
    ("ทวีวัฒนา", 13.7620, 100.3730),
    ("ทุ่งครุ", 13.6450, 100.4980),
    ("บางบอน", 13.6780, 100.3870),
]

pdf_geo = pd.DataFrame(district_centers, columns=["district", "latitude", "longitude"])

# 3) merge → ได้ทุกคอลัมน์เดิม + latitude + longitude
pdf_with_geo = pdf_score.merge(pdf_geo, on="district", how="left")

# 4) จัดอันดับตัวอย่าง (ถ้าอยากดู)
print(
    pdf_with_geo.sort_values("Livability_Score_10", ascending=False)
    [["district", "Livability_Score_10", "Total_Problem_Count", "Avg_Pending_Days"]]
    .head(5)
)

# 5) เซฟออก CSV
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_PATH = PROCESSED_DIR / "traffy_with_livability.csv"

pdf_with_geo.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print("✅ Saved CSV to:", OUTPUT_PATH)


      district  Livability_Score_10  Total_Problem_Count  Avg_Pending_Days
19    ทวีวัฒนา                 9.73                   17          2.588235
16     ทุ่งครุ                 9.06                   22          2.681818
7      หนองจอก                 8.49                   25          2.840000
15  บางกอกใหญ่                 8.23                   21          3.047619
1     ดอนเมือง                 8.13                   25          3.000000
✅ Saved CSV to: C:\Users\sasit\CU\2-1\dsde\project\dsdengdeng-project-dsde\data\processed\traffy_with_livability.csv


In [56]:
import pandas as pd
from pymongo import MongoClient
from dotenv import load_dotenv
import os

load_dotenv()

# 1) Connect
MONGO_URI = os.getenv("MONGO_URI")
client = MongoClient(MONGO_URI)

db = client["my_project"]
collection = db["district_livability"]   # แนะนำให้ใช้ collection แยกจาก traffic_clean

# 2) อ่านไฟล์ CSV (50 แถว)
df = pd.read_csv(str(OUTPUT_PATH))

print(f"📌 Loaded {len(df)} rows from CSV")

# 3) ลบข้อมูลเก่าทั้งหมด
delete_result = collection.delete_many({})
print(f"🗑️ Deleted {delete_result.deleted_count} old documents")

# 4) Insert ใหม่ทั้งหมด
data_dict = df.to_dict("records")
collection.insert_many(data_dict)

print(f"✅ Inserted {len(data_dict)} new rows into MongoDB")


📌 Loaded 50 rows from CSV
🗑️ Deleted 50 old documents
✅ Inserted 50 new rows into MongoDB
